# 02 · 첫 모델 — U-Net 베이스라인 학습하기

> **공부 기록 노트북 2번.** 01에서 데이터를 이해했으니, 이제 **첫 모델**을
> 직접 학습시켜 봅니다. 목표는 좋은 모델을 얻는 게 아니라
> *"분할 모델이 어떻게 학습되는가"* 를 두 눈으로 보는 것입니다.

## 01 복습

- CholecSeg8k = 수술 프레임 + 픽셀별 6-class 정답 마스크
- 클래스 불균형이 심하다 (배경/간이 대부분, 담낭관은 극소수)
- train/val/test 는 영상 단위로 나눈다

## 이 노트북에서 할 일

1. **베이스라인**이 왜 필요한지 이해
2. **U-Net** 분할 모델의 구조 이해 (그림으로)
3. 모델 · 데이터 · 손실(loss) 을 하나씩 만들어 보기
4. **학습 루프**를 직접 돌려 loss 가 줄어드는 걸 확인
5. 학습 전/후 예측을 눈으로 비교

⚠️ **GPU 런타임이 필요합니다** — Runtime → Change runtime type → T4 GPU.

이 노트북의 학습은 *미니 데모*입니다 (데이터 일부 · 3 epoch). 진짜 전체
학습은 `src/train/train_segmentation.py`가 합니다 — 여기서는 원리만 봅니다.

## 0. 환경 준비

01과 동일합니다 — Colab은 매번 빈 컴퓨터이므로 코드를 내려받고 라이브러리를
설치합니다. 이미 돼 있으면 최신 코드만 당겨오므로 **여러 번 실행해도 안전**.

> 키워드: git clone, pip install, GPU 런타임

In [ ]:
%cd /content
import os
if not os.path.isdir("surgical-ai"):
    !git clone -b main https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git pull --ff-only
!bash scripts/colab_setup.sh

import torch
print("준비 완료 ·", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "없음 → Runtime > Change runtime type > GPU 로 바꾸세요")

### (선택) Google Drive 연결 — 다운로드·체크포인트 보존

Colab은 런타임이 초기화되면 `/content`가 비워져 데이터(3GB+)와 학습 체크포인트가
사라집니다. 아래 셀은 Drive를 마운트해 `data/`·`outputs/`·모델 캐시를 Drive에
저장하므로, **다음에 다시 와도 받아둔 데이터와 학습 결과를 그대로** 씁니다.

**기본값은 `False`(Drive 미사용)** — 전부 `/content` 에서 돌아갑니다. Drive 에 보존하고 싶을 때만 `USE_DRIVE = True` 로 바꾸세요(인증 창이 뜹니다).

In [ ]:
USE_DRIVE = False  # Drive 에 데이터·체크포인트를 보존하려면 True (Drive 없으면 그대로 두세요)
if USE_DRIVE:
    from src.utils.colab import mount_drive
    mount_drive()

## 1. "베이스라인"이 왜 필요한가

이 프로젝트의 주력 모델은 SAM2(노트북 04)지만, 그 전에 **평범한 모델**을
먼저 학습시킵니다. 이걸 **베이스라인(baseline)**이라 합니다.

왜? "SAM2가 점수 0.80을 받았다"는 그 자체로는 의미가 없습니다. *무엇과
비교해서* 좋은지 알아야 합니다. 평범한 모델이 0.78인데 SAM2가 0.80이면
이득이 작고, 평범한 모델이 0.50인데 0.80이면 큰 이득이죠.
**베이스라인 = 비교의 기준선**입니다. 베이스라인 없는 점수는 못 믿습니다.

## 2. U-Net 구조 — 왜 "U" 인가

분할은 두 가지를 동시에 해야 합니다:
- **무엇**이 있는가 (이건 간? 쓸개?)
- **어디에** 있는가 (정확히 어느 픽셀?)

U-Net은 이를 인코더–디코더로 풉니다:

```
        입력 이미지 (3채널 RGB)
              |
   [ 인코더 ]  이미지를 점점 작게 압축하며
     v v v     "무엇이 있는지" 추출 (위치는 흐려짐)
              |
     skip ----+----  인코더 각 단계를 디코더로 '지름길' 연결
              |       (잃어버린 위치 정보를 직접 전달)
   [ 디코더 ]  다시 점점 크게 복원하며
     ^ ^ ^     "어디에 있는지" 그려냄
              |
     픽셀별 예측 (6채널 — 클래스마다 1장)
```

가운데가 잘록한 **U자 모양**이라 U-Net입니다. **skip connection**(지름길)이
핵심으로, 압축하며 잃은 *세밀한 위치 정보*를 디코더에 직접 전달합니다.

인코더로는 **EfficientNet-B4**(ImageNet으로 미리 학습된 것)를 씁니다. 이미
"사물의 일반적 모양"을 아는 인코더에서 출발하면 학습이 빨라집니다 —
이를 **전이학습(transfer learning)**이라 합니다.

> 키워드: U-Net, encoder-decoder, skip connection, transfer learning,
> EfficientNet

In [ ]:
from src.models.unet_baseline import UNetBaseline

model = UNetBaseline(encoder="efficientnet-b4",
                     encoder_weights="imagenet",   # ImageNet 사전학습 가중치
                     num_classes=6)
n_params = sum(p.numel() for p in model.parameters())
print(f"U-Net 베이스라인 생성 완료 · 파라미터 {n_params / 1e6:.1f}M 개")

## 3. 데이터를 모델이 먹을 수 있는 형태로

모델은 PIL 이미지를 못 먹습니다 — **숫자 텐서(tensor)**가 필요합니다.

```
PIL 이미지 (854x480)
   |  transform : 크기 조정 + 정규화 + (학습용은) 증강
   v
텐서 (3, 256, 256)            <- (채널, 높이, 너비)
   |  DataLoader : 여러 장을 묶음(batch)으로
   v
배치 텐서 (4, 3, 256, 256)     <- 한 번에 4장
```

- **transform** (`src/data/transforms.py`) — 크기를 256으로 맞추고, 픽셀값을
  정규화하고, 학습용은 좌우 뒤집기 등 *증강(augmentation)*을 더합니다.
- **DataLoader** — 데이터를 batch 단위로 모델에 흘려보내는 컨베이어 벨트.

여기서는 *미니 데모*이므로 train 데이터 중 **160장만**, 256x256 크기로 씁니다.

> 키워드: tensor, transform, normalization, data augmentation, DataLoader, batch

In [ ]:
from torch.utils.data import DataLoader, Subset

from src.data.cholecseg8k import CholecSeg8kDataset, CLASS_NAMES
from src.data.transforms import build_train_transforms, build_eval_transforms

IMG = 256
full_train = CholecSeg8kDataset(split="train", image_size=IMG,
                                transform=build_train_transforms(IMG))
demo_train = Subset(full_train, range(160))          # 미니 데모: 160장만
loader = DataLoader(demo_train, batch_size=4, shuffle=True, num_workers=2)

batch = next(iter(loader))
print("배치 image:", tuple(batch["image"].shape), "  <- (B, 3, H, W)")
print("배치 mask :", tuple(batch["mask"].shape), "      <- (B, H, W) 정수 0~5")
print("6-class:", CLASS_NAMES)

## 4. 학습 전 — 모델은 아무것도 모른다

방금 만든 모델의 가중치는 (인코더 빼고) 무작위입니다. 그래서 지금 예측을
시켜 보면 **엉망**입니다. 이건 실패가 아니라 *출발점*입니다 — 나중에 학습
후와 비교할 "before" 사진을 찍어 두는 것입니다.

아래 셀은 학습 전/후 비교용으로 **고정된 한 장**(val 세트의 첫 프레임)을
확보하고, 학습 전 예측을 만들어 둡니다.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# 학습 전/후를 같은 그림으로 비교하려고 고정 샘플 하나 확보 (val, 증강 없음)
eval_ds = CholecSeg8kDataset(split="val", image_size=IMG,
                             transform=build_eval_transforms(IMG))
fixed = eval_ds[0]
fixed_image = fixed["image"].unsqueeze(0).to(device)   # (1, 3, 256, 256)
fixed_mask = fixed["mask"]                             # (256, 256)

model.eval()
with torch.no_grad():
    pred_before = model(fixed_image).argmax(dim=1)[0].cpu()
print("학습 전 예측 생성 완료 (뒤에서 학습 후와 비교합니다)")

## 5. 모델은 어떻게 "배우는가" — loss 와 경사하강법

**loss(손실)** = 예측이 정답에서 *얼마나 틀렸는지*를 나타내는 숫자 하나.
작을수록 잘 맞춘 것. **학습이란 loss 를 줄이는 과정**입니다.

어떻게 줄이나? **경사하강법(gradient descent):**

> 안개 낀 산에서 가장 낮은 골짜기로 내려가야 합니다. 한 치 앞도 안 보이지만
> *발밑의 경사*는 느낄 수 있습니다. 가장 가파른 내리막으로 한 걸음, 또 한
> 걸음 — 반복하면 골짜기에 닿습니다.

"산"이 loss, "한 걸음"이 모델 가중치 업데이트입니다. 경사를 **gradient**,
걸음 크기를 **learning rate(학습률)** 라 합니다.

우리는 loss 두 개를 더해 씁니다:
- **Dice loss** — 예측 영역과 정답 영역이 *얼마나 겹치는지*. 분할 품질에
  직결됩니다.
- **Focal loss** — 똑똑한 분류 loss. *맞추기 쉬운 픽셀(배경)*은 덜 보고
  *어렵고 드문 픽셀(담낭관)*에 집중 → 01에서 본 클래스 불균형에 대응.

> 키워드: loss function, gradient descent, learning rate, Dice loss, Focal loss

In [ ]:
from src.losses.dice import DiceLoss
from src.losses.focal import FocalLoss

dice = DiceLoss()
focal = FocalLoss(gamma=2.0)

target = fixed_mask.unsqueeze(0).to(device)
model.eval()
with torch.no_grad():
    logits = model(fixed_image)
    loss_before = focal(logits, target) + dice(logits, target)
print(f"학습 전 loss: {loss_before.item():.4f}   (높을수록 못 맞춘 것)")

## 6. 학습 루프 — 직접 돌려보기

학습 한 스텝은 언제나 같은 4단계입니다:

```
  1. forward   : batch 를 모델에 넣어 예측을 얻는다
  2. loss      : 예측이 정답에서 얼마나 틀렸는지 계산
  3. backward  : loss 를 줄이려면 각 가중치를 어디로 옮길지(gradient) 계산
  4. step      : 그 방향으로 가중치를 한 걸음 옮긴다
```

이를 batch 마다 반복하고, 데이터를 한 바퀴 다 도는 것을 **1 epoch**이라
합니다. 아래는 3 epoch 미니 학습 — **loss 숫자가 줄어드는지** 지켜보세요.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
EPOCHS = 3

model.train()
for epoch in range(EPOCHS):
    for step, batch in enumerate(loader):
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)

        logits = model(images)                              # 1. forward
        loss = focal(logits, masks) + dice(logits, masks)   # 2. loss
        optimizer.zero_grad()
        loss.backward()                                     # 3. backward
        optimizer.step()                                    # 4. step

        if step % 10 == 0:
            print(f"epoch {epoch + 1}/{EPOCHS}  step {step:2d}  "
                  f"loss {loss.item():.4f}")
print("미니 학습 완료")

## 7. 배웠을까? — 학습 전/후 비교

이제 4단계를 여러 번 거쳤으니, 처음의 그 *고정된 한 장*을 다시 예측시켜
학습 전과 비교합니다. 미니 데모(160장 · 3 epoch)라 완벽하진 않지만, 학습
후 예측이 정답 쪽으로 **눈에 띄게 다가갔다면** 학습 루프가 제대로 돈
것입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    out = model(fixed_image)
    pred_after = out.argmax(dim=1)[0].cpu()
    loss_after = focal(out, target) + dice(out, target)

print(f"loss  학습 전 {loss_before.item():.4f}  ->  학습 후 {loss_after.item():.4f}")

# 입력 이미지는 정규화돼 있으므로 0~1로 되돌려 표시
img = fixed_image[0].cpu().permute(1, 2, 0).numpy()
img = (img - img.min()) / (img.max() - img.min() + 1e-8)

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow(img);                                       ax[0].set_title("input")
ax[1].imshow(fixed_mask, vmin=0, vmax=5, cmap="tab10");  ax[1].set_title("ground truth")
ax[2].imshow(pred_before, vmin=0, vmax=5, cmap="tab10"); ax[2].set_title("before training")
ax[3].imshow(pred_after, vmin=0, vmax=5, cmap="tab10");  ax[3].set_title("after training")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## 마무리 — 이 노트북 정리

### 이 노트북에서 배운 것
- **베이스라인**은 "비교 기준". 베이스라인 없는 점수는 의미를 판단할 수 없다.
- **U-Net** = 인코더(무엇인지 압축) + 디코더(어디인지 복원) + skip
  connection(위치 정보 지름길). 가운데가 잘록해 "U".
- 데이터는 transform → tensor → DataLoader(batch) 를 거쳐 모델에 들어간다.
- **학습 = loss 를 줄이는 것.** loss 는 "얼마나 틀렸나"는 숫자 하나이고,
  **경사하강법**으로 한 걸음씩 줄인다.
- 학습 한 스텝 = **forward → loss → backward → step** 4단계의 반복.
- Dice loss(겹침) + Focal loss(어려운 픽셀 집중) 를 함께 쓴다.

### 아직 이해가 덜 된 것 / 더 파볼 것
- learning rate(`lr=1e-3`)는 어떻게 정하나? 너무 크면/작으면?
- epoch 을 몇 번 돌려야 하나? 너무 많이 돌리면 어떻게 되나 (overfitting)?
- 미니 데모 말고 진짜 학습은? → `src/train/train_segmentation.py` 와
  `SegmentationModule`(PyTorch Lightning)이 검증·체크포인트·스케줄러까지
  포함해 처리한다. 한 번 열어 보기.
- 성능을 *수치로* 어떻게 재나? (mIoU, Dice 점수) → 노트북 03에서 다룬다.

### 다음 노트북에서 할 것
**`03_sam2_zeroshot.ipynb`** — 직접 학습시키지 않은 거대 모델 **SAM2**가
수술 영상을 *그냥* 얼마나 분할하는지(zero-shot) 본다. 그리고 분할 품질을
**숫자(IoU)**로 재는 법을 배운다 — U-Net 베이스라인과 비교할 기준이 된다.